In [25]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_score
from reader import PickledReviewsReader
from transformer import TextNormalizer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
def documents(corpus):
    return list(corpus.reviews())

def continuous(corpus):
    return list(corpus.scores())

def make_categorical(corpus):
    return np.digitize(continuous(corpus), [0.0, 3.0, 5.0, 7.0, 10.1])

In [27]:
def train_model(path, model, cts=True, saveto=None, cv=12):
    """
    Trains model from corpus at specified path; constructing cross-validation
    scores using the cv parameter, then fitting the model on the full data.
    Returns the scores.
    """
    # Load the corpus data and labels for classification
    corpus = PickledReviewsReader(path)
    X = documents(corpus)
    if cts:
        y = continuous(corpus)
        scoring = 'r2'
    else:
        y = make_categorical(corpus)
        scoring = 'f1'
    # Compute cross-validation scores
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    # Fit the model on entire dataset
    model.fit(X, y)
    # Return scores
    return scores

In [29]:
def main():
    # Path to postpreprocessed, part-of-speech tagged review corpus
    cpath = 'C:/Users/PS3ma/Documents/Bellevue University/DSC 550/target.json'

    regressor = Pipeline([
                        ('norm', TextNormalizer()),
                        ('tfidf', TfidfVectorizer()),
                        ('ann', MLPRegressor(hidden_layer_sizes=[500,150], verbose=True))
                        ])

    regression_scores = train_model(cpath, regressor, cts = True)

    classifier = Pipeline([
                         ('norm', TextNormalizer()),
                         ('tfidf', TfidfVectorizer()),
                         ('ann', MLPClassifier(hidden_layer_sizes=[500,150], verbose=True))
                         ])

    classifer_scores = train_model(cpath, classifier, cts = False)

In [30]:
if __name__ == '__main__':
    main()

ValueError: Cannot have number of splits n_splits=12 greater than the number of samples: n_samples=0.